In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_parquet('../data/processed/rossmann_clean.parquet', engine='fastparquet')
print('Shape: ', df.shape)
print('Dtypes: \n', df.dtypes)

df['Date'] = pd.to_datetime(df['Date'])
print('\n Date Range: ', df['Date'].min(), 'to', df['Date'].max())

# How many stores report daily
daily_counts = df.groupby('Date').size()
print('\nDaily counts: \n', daily_counts.describe())

print('\nSome of the lowest count days: \n', daily_counts.sort_values().head(20))

Shape:  (844338, 23)
Dtypes: 
 Store                                 int64
DayOfWeek                             int64
Date                         datetime64[us]
Sales                                 int64
Customers                             int64
Open                                  int64
Promo                                 int64
StateHoliday                       category
SchoolHoliday                         int64
StoreType                          category
Assortment                         category
CompetitionDistance                 float64
CompetitionOpenSinceMonth             int64
CompetitionOpenSinceYear              int64
Promo2                                int64
Promo2SinceWeek                       int64
Promo2SinceYear                       int64
PromoInterval                      category
CompetitionInfoMissing                int64
Year                                  int32
Month                                 int32
Day                                   int32
W

In [4]:
monthly_stores = df.groupby(df['Date'].dt.to_period('M'))['Store'].nunique()
print('Distinct stores reporting each month: \n', monthly_stores.to_string())

Distinct stores reporting each month: 
 Date
2013-01    1112
2013-02    1112
2013-03    1112
2013-04    1112
2013-05    1113
2013-06    1113
2013-07    1115
2013-08    1115
2013-09    1115
2013-10    1115
2013-11    1115
2013-12    1115
2014-01    1115
2014-02    1115
2014-03    1115
2014-04    1115
2014-05    1115
2014-06    1114
2014-07     934
2014-08     934
2014-09     934
2014-10     935
2014-11     935
2014-12     935
2015-01    1115
2015-02    1115
2015-03    1115
2015-04    1115
2015-05    1115
2015-06    1115
2015-07    1115
Freq: M


After the `Open==1` filter from previous notebook, daily row counts collapse on
Sundays/holidays (e.g. 16 rows on Christmas = 16 stores open), so row-count-per-day can't distinguish "store closed" from "store not reporting". Counting **distinct stores per month** resolves this; closures change rows-per-store, not the count of distinct stores in a month.

**Finding:** Store coverage holds at ~1,115 except 2014-07 → 2014-12, where it
drops to ~934, roughly 181 stores absent for ~6 months, then a clean recovery
to 1,115 in 2015-01. The sharp, synchronized drop-and-recovery indicates a
**data-collection artifact**, not a real business event.

**Implication for modeling:**
- The gap sits entirely in second half of 2014, deep inside training history; so we can validate on the last 6-weeks of dataset and train on the remaining.
- All of 2015 is complete, so a time-based validation window in mid-2015 is unaffected; every store is represented in validation.

- **Decision:** 

Keep the gap in the training set (cannot recover missing rows); it does not distort the validation score. Flag it: the ~181 affected stores have thinner 2014 H2 history.

In [5]:
max = df['Date'].max()
val_start = max - pd.Timedelta(days=41)

train = df[df['Date'] < val_start].copy()
val = df[df['Date'] >= val_start].copy()

print('Cutoff Date: ', val_start.date())
print('\nTrain range: ', train['Date'].min().date(), 'to', train['Date'].max().date())
print('Validation range: ', val['Date'].min().date(), 'to', val['Date'].max().date())
print('\nTrain Rows: ', len(train), "and Validation Rows: ", len(val))
print('Validation proportion: {:.2%}'.format(len(val) / len(df)))
print('\nValidation distinct days:', val['Date'].nunique(), 'and Validation distinct stores:', val['Store'].nunique())

Cutoff Date:  2015-06-20

Train range:  2013-01-01 to 2015-06-19
Validation range:  2015-06-20 to 2015-07-31

Train Rows:  804056 and Validation Rows:  40282
Validation proportion: 4.77%

Validation distinct days: 42 and Validation distinct stores: 1115


Forecasting task + strong yearly seasonality means validation must: train on history, validate on the most recent unseen window. A random split would leak future information backward.

Last **42 days** (6 full weeks) as validation. 42 being multiple of 7, so each weekday appears 6× in validation; weekday balance is preserved.

Validation contains all 42 distinct days, including Sundays — a small number of stores trade on Sundays (near-zero sales). These low-sales days are in validation, which informs the choice of error metric (avoid metrics distorted by many small-denominator rows).

In [6]:
target = 'Sales'
not_features = ['Sales', 'Customers', 'Date',]

feature_cols = [c for c in df.columns if c not in not_features]

print('Target: ', target)
print('\nDropped from Features: \n', not_features)
print('\nFeature Columns: \n', feature_cols)

assert 'Customers' not in feature_cols
assert 'Date' not in feature_cols
assert 'Sales' not in feature_cols
print('\nAll checks passed!')

Target:  Sales

Dropped from Features: 
 ['Sales', 'Customers', 'Date']

Feature Columns: 
 ['Store', 'DayOfWeek', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval', 'CompetitionInfoMissing', 'Year', 'Month', 'Day', 'WeekOfYear']

All checks passed!


In [7]:
print('Open values: ', train['Open'].nunique())
print('\nStateHoliday Distribution: \n', train['StateHoliday'].value_counts())

Open values:  1

StateHoliday Distribution: 
 StateHoliday
0    803146
a       694
b       145
c        71
Name: count, dtype: int64


In [8]:
not_features = ['Sales', 'Customers', 'Date', 'Open']
feature_cols = [c for c in df.columns if c not in not_features]

print('Feature Columns after dropping Open: \n', feature_cols)

assert 'Open' not in feature_cols
print('\nAll checks passed!')

Feature Columns after dropping Open: 
 ['Store', 'DayOfWeek', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval', 'CompetitionInfoMissing', 'Year', 'Month', 'Day', 'WeekOfYear']

All checks passed!


### Feature set definition 

**Target:** `Sales`

**Excluded from features:**
| Column | Reason |
|---|---|
| `Sales` | Target. |
| `Customers` | **Leakage.** Co-produced with Sales, known only after the day ends, never available at forecast time. |
| `Date` | Time axis; information already extracted to Year/Month/Day/WeekOfYear. |
| `Open` | **Constant** after the `Open==1` filter. Zero information. Closed days are handled as Sales=0 outside the model. |

**Kept despite being near-constant:**
- `StateHoliday` — only 0.11% of the total rows are non-zero here after the
  `Open==1 & Sales>0` filter. Kept because those rows are **hypothesized** to cluster in StoreType 'b', where holiday-trading may be part of their outlier sales. Won't move the aggregate metric; preserved at zero leakage cost for its store-type-specific signal.

**Final feature set: 19 columns.**

In [9]:
# Converting categorical columns to category dtype
cat_cols = ['Store', 'StoreType', 'Assortment', 'StateHoliday', 'PromoInterval']

store_cat = pd.CategoricalDtype(
    categories=sorted(pd.concat(
        [train['Store'], val['Store']]
    ).unique())
)

train['Store'] = train['Store'].astype(store_cat)
val['Store'] = val['Store'].astype(store_cat)

for c in cat_cols:
    print(f'{c:15s} train={str(train[c].dtype):10s} val={str(val[c].dtype):10s}' f'n_cats={train[c].nunique()}')

train_cats = [c for c in train.columns if str(train[c].dtype) == "category"]
print("\nCategory-dtype columns in train:", train_cats)

Store           train=category   val=category  n_cats=1115
StoreType       train=category   val=category  n_cats=4
Assortment      train=category   val=category  n_cats=3
StateHoliday    train=category   val=category  n_cats=4
PromoInterval   train=category   val=category  n_cats=4

Category-dtype columns in train: ['Store', 'StateHoliday', 'StoreType', 'Assortment', 'PromoInterval']


### LightGBM (native Categorical handling)

**Model:** LightGBM (gradient-boosted trees). Chosen over XGBoost for cleaner
native categorical support; given `Store` has 1,115 labels.

- One-hot on `Store` → 1,115 new columns, which fragments tree splits and degrades performance. One-hot is a linear-model tool; wrong for trees here.

**Approach:** Mark nominal categoricals as pandas `category` dtype; LightGBM
auto-detects and finds optimal category groupings per split (no fake numeric order).

**Categorical columns (5):** `Store` (converted int → category, the fix for IDs being misread as quantities), `StoreType`, `Assortment`, `StateHoliday`, `PromoInterval` (latter four already category-dtype).

- `Store` cardinality (1,115) is high for native categorical handling. We'll watch for overfitting on rare stores once the model is trained.

In [14]:
# metric function
def rmspe(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    mask = y_true > 0
    pct_error = (y_true[mask] - y_pred[mask]) / y_true[mask]
    return np.sqrt(np.mean(pct_error ** 2))

def rmse(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

In [15]:
# Baseline for predicting avg training data sale per (sotre/day-of-week)
bl_table = train.groupby(['Store', 'DayOfWeek'], observed=True)['Sales'].mean()
global_mean = train['Sales'].mean()

# for each val row, look up bl_table
val_keys = list(zip(val['Store'], val['DayOfWeek']))
bl_pred = np.array([bl_table.get(k, global_mean) for k in val_keys])

n_fallback = sum(k not in bl_table.index for k in val_keys)
print('Val rows: ', len(val))
print('Fallback rows (store/day-of-week not in training): ', n_fallback)

#score baseline (LightGBM will need to beat this)
print('\nBaseline RMSPE: ', round(rmspe(val['Sales'].values, bl_pred), 4))
print('\nBaseline RMSE: ', round(rmse(val['Sales'].values, bl_pred), 2))

Val rows:  40282
Fallback rows (store/day-of-week not in training):  0

Baseline RMSPE:  0.2356

Baseline RMSE:  1664.77


In [17]:
import lightgbm as lgb
x_train, y_train = train[feature_cols], train[target]
x_val, y_val = val[feature_cols], val[target]

print('X Train: ', x_train.shape, 'and X Val:', x_val.shape)
print('Categorical dtypes LightGBM will auto-detect: ', [c for c in feature_cols if str(x_train[c].dtype) == 'category'])


X Train:  (804056, 19) and X Val: (40282, 19)
Categorical dtypes LightGBM will auto-detect:  ['Store', 'StateHoliday', 'StoreType', 'Assortment', 'PromoInterval']


In [18]:
model = lgb.LGBMRegressor(
    objective='regression',
    n_estimators=1000,
    learning_rate=0.05,
    random_state=42,
    n_jobs=1,
)

model.fit(
    x_train, y_train,
    eval_set=[(x_val, y_val)],
    eval_metric='rmse',
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=100)],
)

# predict on val and score with our metrics
lgb_pred = model.predict(x_val)

print("\n--- LightGBM vs Baseline ---")
print("LightGBM RMSPE:", round(rmspe(y_val.values, lgb_pred), 4), " | Baseline RMSPE: 0.2356")
print("LightGBM RMSE:", round(rmse(y_val.values, lgb_pred), 2), " | Baseline RMSE : 1664.77")
print("\nBest iteration (trees used):", model.best_iteration_)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.078007 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1561
[LightGBM] [Info] Number of data points in the train set: 804056, number of used features: 19
[LightGBM] [Info] Start training from score 6954.822954
Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 1163.83	valid_0's l2: 1.35451e+06
[200]	valid_0's rmse: 1041.45	valid_0's l2: 1.08462e+06
[300]	valid_0's rmse: 1000.41	valid_0's l2: 1.00083e+06
[400]	valid_0's rmse: 975.594	valid_0's l2: 951783
[500]	valid_0's rmse: 957.048	valid_0's l2: 915940
[600]	valid_0's rmse: 936.34

> Increasing the number of trees so that model can finish learning before running out of trees.

In [19]:
model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=3000,           # raised cap so early stopping can trigger
    learning_rate=0.05,          
    random_state=42,
    n_jobs=-1,
)

model.fit(
    x_train, y_train,
    eval_set=[(x_val, y_val)],
    eval_metric="rmse",
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=200)],
)

lgb_pred = model.predict(x_val)

print("\n--- LightGBM (full fit) vs Baseline ---")
print("LightGBM RMSPE:", round(rmspe(y_val.values, lgb_pred), 4), " | Baseline: 0.2356")
print("LightGBM RMSE :", round(rmse(y_val.values, lgb_pred), 2),  " | Baseline: 1664.77")
print("Best iteration:", model.best_iteration_, "(cap was 3000)")

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.051968 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1561
[LightGBM] [Info] Number of data points in the train set: 804056, number of used features: 19
[LightGBM] [Info] Start training from score 6954.822954
Training until validation scores don't improve for 50 rounds
[200]	valid_0's rmse: 1041.45	valid_0's l2: 1.08462e+06
[400]	valid_0's rmse: 975.594	valid_0's l2: 951783
[600]	valid_0's rmse: 936.343	valid_0's l2: 876738
[800]	valid_0's rmse: 898.241	valid_0's l2: 806836
[1000]	valid_0's rmse: 888.863	valid_0's l2: 790077
[1200]	valid_0's rmse: 878.408	valid_

Early stopping triggered at iteration 1714, well under the 3000 cap. That's the message we wanted: the model improved, plateaued for 50 rounds, and LightGBM rolled back to its genuine best. So 1714 trees is where an untuned LightGBM naturally converges on this problem, that's real information, and it's the anchor the tuning notebook starts from.

### Modeling results (LightGBM vs naive baseline)

**Metric:** RMSPE primary (size-fair as error is divided by actual sales, ignores zero-actual rows), RMSE as guard. 

**Baseline** (predict avg train Sales per Store/DayOfWeek; 0 fallback rows):
- RMSPE 0.2356 | RMSE 1664.77

**LightGBM** (untuned; objective=L2, lr=0.05, early stopping → 1,714 trees):
- RMSPE **0.1277** | RMSE **870.98**
- ~46% RMSPE reduction vs baseline; both metrics reduce together → uniform improvement across store sizes, not a large-store/small-store tradeoff.

**Status: untuned floor, not ceiling.** Known unused levers:
1. `Store` high-cardinality warning firing; 1,115 categories handled
   approximately by LightGBM defaults (needs max_cat_threshold/min_data_per_group).
2. Objective/metric mismatch — trained on L2, measured RMSPE
3. No hyperparameter tuning yet.

**Caveat (warning attached to a condition):** Single validation window (last 6 weeks). Trustworthy for baseline comparison; not yet validated for stability across time folds.